# 9.3 · PyTorch 入门 / PyTorch Basics

> **课程定位 / Where this fits**
> 第 3 课，**Part 9 · 深度学习基础**。
> Lesson 3, **Part 9 · Deep Learning Foundations**.
>
> 9.2 手写了前向+反向+SGD，体会到了"算梯度"的繁琐。**PyTorch** 让这一切自动化：**autograd 自动微分**替你算所有梯度，`nn.Module` 帮你搭网络，`DataLoader` 帮你喂数据。PyTorch 是当前**学术界和工业界的主流深度学习框架**，会用它是深度学习岗的硬技能。
> 9.2 hand-coded forward+backward+SGD and felt the tedium of computing gradients. **PyTorch** automates it all: **autograd** computes every gradient for you, `nn.Module` builds nets, `DataLoader` feeds data. PyTorch is the **dominant DL framework in research and industry** — a hard skill for DL roles.
>
> 💼 **实战/面试视角**："autograd 怎么工作 / 训练循环五件套 / train vs eval 模式 / 为什么要 zero_grad" 必问。
> 💼 **Practical/interview angle:** "how autograd works / the 5-step training loop / train vs eval / why zero_grad" — must-knows.

> 📐 **符号约定 / Notation**
> - tensor —— PyTorch 的多维数组(带梯度追踪)/ multi-dim array with gradient tracking
> - `requires_grad` —— 是否追踪梯度 / whether to track gradients
> - 计算图 computational graph —— autograd 记录的运算图 / the graph autograd records

> 💡 **面试相关 / Interview-relevant**
> - "autograd 自动微分原理(计算图)"（出镜率 ★★★★★）
> - "训练循环的五个步骤"（出镜率 ★★★★★）
> - "为什么每步要 optimizer.zero_grad()"（★★★★★，梯度累加）
> - "model.train() 和 model.eval() 区别"（★★★★）
> - "torch.no_grad() 干什么"（★★★★）

---

## 学习目标 / Learning Objectives

1. 掌握 tensor 基础与 **autograd 自动微分**。
   Master tensor basics and **autograd**.
2. 用 **nn.Module** 定义网络。
   Define networks with **nn.Module**.
3. 用 **Dataset/DataLoader** 批量喂数据。
   Feed data in batches via Dataset/DataLoader.
4. 写出标准**训练循环的五件套**（含为什么 zero_grad）。
   Write the standard 5-step training loop (and why zero_grad).
5. 区分 **train/eval 模式** 与 `torch.no_grad()`。
   Distinguish train/eval modes and `torch.no_grad()`.

## 目录 / TOC
1. [先建直觉 + autograd ⭐](#1)
2. [autograd 验证：和手算梯度对比 ⭐](#2)
3. [nn.Module 定义网络 ⭐](#3)
4. [🔢 DataLoader + 训练循环五件套 ⭐](#4)
5. [train/eval + no_grad + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + autograd ⭐ / Intuition & Autograd

9.2 里我们**手推每个梯度**，繁琐又易错。PyTorch 的核心魔法是 **autograd（自动微分）**：你只管写前向计算，PyTorch 会**自动记录一张计算图**（哪些张量经过哪些运算得到结果），然后调一次 `.backward()`，它就**沿计算图反向自动应用链式法则**，把所有梯度算出来。
In 9.2 we **derived every gradient by hand** — tedious and error-prone. PyTorch's core magic is **autograd**: you only write the forward computation, and PyTorch **records a computational graph automatically** (which tensors went through which ops); one call to `.backward()` then **applies the chain rule backward over the graph automatically**, computing all gradients.

关键概念：给张量设 `requires_grad=True`，PyTorch 就开始追踪它的所有运算。`.backward()` 后，梯度存在 `.grad` 里。**你写的还是 9.2 那套链式法则，只是 PyTorch 替你执行了反向那一趟。**
Key concept: set `requires_grad=True` and PyTorch tracks all operations on that tensor. After `.backward()`, gradients live in `.grad`. **It's still the chain rule from 9.2 — PyTorch just runs the backward pass for you.**


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
torch.manual_seed(0)

# 最小例子: 手算 vs autograd / a minimal autograd example
x = torch.tensor(3.0, requires_grad=True)        # 追踪这个张量的梯度
y = x**2 + 2*x + 1                                # 前向: y = x²+2x+1, PyTorch 自动记录计算图
y.backward()                                      # 反向: 自动算 dy/dx
print(f"y = x²+2x+1 在 x=3: y={y.item()}")
print(f"autograd 算的 dy/dx = {x.grad.item()}  (解析: 2x+2 = {2*3+2}) → 一致")
print("\nautograd: 只写前向, .backward() 自动沿计算图反向应用链式法则算梯度")


<a id="2"></a>
## 2. autograd 验证：和手算梯度对比 ⭐ / Verifying Autograd vs Hand-derived

为了证明 autograd 做的就是 9.2 手写的反向传播，我们用 PyTorch 张量重写 9.2 的两层网络的**前向**，然后让 autograd 算梯度，和 9.2 手推的公式（如 $\frac{\partial L}{\partial W_2}=a_1^\top(p-y)/n$）对比——应完全一致。
To prove autograd does exactly the backprop we hand-wrote in 9.2, we re-implement that two-layer net's **forward** with PyTorch tensors, let autograd compute gradients, and compare to 9.2's hand-derived formulas (e.g. $\frac{\partial L}{\partial W_2}=a_1^\top(p-y)/n$) — they match.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.data / 16.0
X_tr, X_te, y_tr, y_te = train_test_split(X, digits.target, test_size=0.3, stratify=digits.target, random_state=0)

# 用 torch 张量手写前向, 让 autograd 算梯度 / forward in torch, autograd for backward
Xt = torch.tensor(X_tr[:50], dtype=torch.float32)
yt = torch.tensor(y_tr[:50])
W1 = torch.randn(64, 32, requires_grad=True) * 0.1; W1.retain_grad()
W2 = torch.randn(32, 10, requires_grad=True) * 0.1; W2.retain_grad()

z1 = Xt @ W1; a1 = torch.relu(z1); z2 = a1 @ W2           # 前向(同 9.2)
loss = torch.nn.functional.cross_entropy(z2, yt)         # softmax + 交叉熵
loss.backward()                                           # autograd 自动算所有梯度

# 手算 W2 的梯度(9.2 公式): a1ᵀ(p-y)/n / hand-derived gradient for comparison
probs = torch.softmax(z2, dim=1).detach()
onehot = torch.eye(10)[yt]
manual_dW2 = (a1.detach().T @ (probs - onehot)) / len(yt)
print(f"autograd vs 手算 dW2 最大差异: {(W2.grad - manual_dW2).abs().max().item():.2e} → 一致")
print("→ autograd 做的就是 9.2 手写的反向传播, 只是自动化了; 你不用再手推梯度")


<a id="3"></a>
## 3. nn.Module 定义网络 ⭐ / Defining Networks with nn.Module

实际中不会手管理 W1/W2，而是用 **`nn.Module`** 封装网络：`nn.Linear` 是带可学习权重的全连接层，它会自动注册参数（autograd 自动追踪）。两种写法：`nn.Sequential`（简单堆叠）或继承 `nn.Module`（灵活，可自定义 forward）。
In practice you don't hand-manage W1/W2; you use **`nn.Module`** to encapsulate the net: `nn.Linear` is a fully-connected layer with learnable weights, auto-registered as parameters (autograd-tracked). Two styles: `nn.Sequential` (simple stacking) or subclassing `nn.Module` (flexible, custom forward).


In [ ]:
import torch.nn as nn

# 写法一: Sequential(简单堆叠) / style 1: Sequential
net_seq = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

# 写法二: 继承 nn.Module(灵活, 可自定义 forward) / style 2: subclass nn.Module
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 64)        # 全连接层, 权重自动注册为参数
        self.fc2 = nn.Linear(64, 10)
    def forward(self, x):                    # 只需定义前向; 反向由 autograd 自动处理
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

net = MLP()
n_params = sum(p.numel() for p in net.parameters())      # 统计可学习参数总数
print(f"MLP 结构:\n{net}")
print(f"\n可学习参数总数: {n_params} (= 64×64+64 + 64×10+10)")
print("nn.Linear 自动管理权重+偏置, autograd 自动追踪 → 只写 forward, 反向全自动")


<a id="4"></a>
## 4. DataLoader + 训练循环五件套 ⭐ / DataLoader & the 5-step Loop

**DataLoader** 帮你把数据自动切成 mini-batch、打乱、（可）多进程加载。**训练循环的五件套**是 PyTorch 的标准范式，必须背熟：
**DataLoader** auto-batches, shuffles, and (optionally) multi-process-loads data. The **5-step training loop** is PyTorch's standard idiom — memorize it:

1. `optimizer.zero_grad()` —— **清零梯度**（PyTorch 默认**累加**梯度，不清就会把上一步的加进来！）。
   Zero the gradients (PyTorch **accumulates** by default; forgetting this adds last step's grads!).
2. `output = model(x)` —— 前向。
   Forward.
3. `loss = loss_fn(output, y)` —— 算损失。
   Compute loss.
4. `loss.backward()` —— autograd 反向算梯度。
   Backward (autograd computes gradients).
5. `optimizer.step()` —— 用梯度更新参数。
   Update parameters using the gradients.

**为什么必须 zero_grad（面试高频）**：PyTorch 的 `.grad` 是**累加**的（这设计是为了支持梯度累积等技巧）。不清零的话，每一步的梯度会叠在上一步上，等于用了错误的(累加的)梯度，训练崩溃。
**Why zero_grad is mandatory** (frequently asked): `.grad` **accumulates** in PyTorch (by design, to support gradient accumulation). Without zeroing, each step's gradient piles onto the previous, using a wrong (summed) gradient and breaking training.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 包成 Dataset → DataLoader(自动分批+打乱) / wrap into a DataLoader
train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
Xte_t = torch.tensor(X_te, dtype=torch.float32)

net = MLP()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

losses = []
for epoch in range(30):
    net.train()                                  # 训练模式(影响 dropout/BN, 见第5节)
    for xb, yb in train_loader:                  # DataLoader 自动给出一个个 mini-batch
        optimizer.zero_grad()                    # ① 清零梯度(否则累加!)
        out = net(xb)                            # ② 前向
        loss = loss_fn(out, yb)                  # ③ 算损失
        loss.backward()                          # ④ autograd 反向算梯度
        optimizer.step()                         # ⑤ 用梯度更新参数
    losses.append(loss.item())

net.eval()                                       # 评估模式
with torch.no_grad():                            # 评估不需要梯度 → 省内存提速
    acc = (net(Xte_t).argmax(1).numpy() == y_te).mean()
print(f"PyTorch MLP on Digits: test 准确率 = {acc:.3f}")
plt.figure(figsize=(6,3)); plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("训练损失")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. train/eval + no_grad + 小结 ⭐ / train/eval, no_grad & Summary

两个易错但重要的开关：
Two error-prone but important switches:
- **`model.train()` vs `model.eval()`**：切换训练/评估模式。它**改变 dropout(9.10) 和 BatchNorm(9.10) 的行为**——训练时 dropout 随机丢神经元、BN 用当前 batch 统计；评估时 dropout 关闭、BN 用训练期累积的统计。**评估前忘了 `eval()` 是经典 bug**（结果随机、不可复现）。
  Switch train/eval mode. It **changes the behavior of dropout (9.10) and BatchNorm (9.10)** — in training dropout drops neurons and BN uses batch stats; in eval dropout is off and BN uses accumulated stats. **Forgetting `eval()` before evaluation is a classic bug** (random, irreproducible results).
- **`torch.no_grad()`**：告诉 autograd "这段不用算梯度"（评估/推理时）。它**关闭计算图记录 → 省内存、提速**。注意它和 `eval()` 是**两件独立的事**：`eval()` 改层行为，`no_grad()` 关梯度追踪——评估时**两个都要**。
  Tells autograd "don't track gradients here" (during eval/inference). It **disables graph recording → saves memory, speeds up**. Note it's **separate from `eval()`**: `eval()` changes layer behavior, `no_grad()` disables gradient tracking — you want **both** at evaluation.


In [ ]:
import time
net.eval()
# no_grad 省内存提速演示 / no_grad saves memory and time
t = time.perf_counter()
with torch.no_grad():
    for _ in range(50): net(Xte_t)
t_nograd = time.perf_counter() - t
t = time.perf_counter()
for _ in range(50): net(Xte_t)                   # 不加 no_grad: 仍在建计算图(浪费)
t_grad = time.perf_counter() - t
print(f"推理 50 次: 加 no_grad {t_nograd*1000:.0f}ms, 不加 {t_grad*1000:.0f}ms → no_grad 更快+省内存")
print("\n两个开关(评估时都要开):")
print("  model.eval(): 改 dropout/BN 行为(关 dropout, BN 用累积统计)")
print("  torch.no_grad(): 关梯度追踪, 省内存提速")
print("评估前忘了 eval() = 经典 bug(dropout/BN 还在训练模式 → 结果随机)")


```
autograd: 只写前向, PyTorch 自动记计算图, .backward() 沿图反向自动算所有梯度(=9.2 的 backprop)
nn.Module: 封装网络(nn.Linear 自动管权重); Sequential(简单) 或 继承 nn.Module(灵活, 写 forward)
DataLoader: 自动分批+打乱+(多进程)加载
训练循环五件套: zero_grad → forward → loss → backward → step
  zero_grad 必须: PyTorch 梯度默认累加, 不清零会叠加上一步 → 训练崩
train()/eval(): 切换 dropout/BN 行为; 评估前必须 eval(), 否则结果随机
no_grad(): 关梯度追踪(评估/推理), 省内存提速; 和 eval() 是两件事, 评估时都要
```

### 💡 面试速查 / Interview cheat-sheet
1. **autograd 自动微分**: 记计算图, .backward() 自动算梯度(=自动化的 backprop)。
   Autograd records a graph; .backward() auto-computes gradients (automated backprop).
2. **训练五件套**: zero_grad → forward → loss → backward → step。
   The 5 steps: zero_grad → forward → loss → backward → step.
3. **必须 zero_grad**: PyTorch 梯度累加, 不清会叠加 → 训练错。
   Must zero_grad: gradients accumulate; otherwise they pile up and training breaks.
4. **train()/eval() 改 dropout/BN 行为**; 评估前忘 eval() 是经典 bug。
   train()/eval() change dropout/BN; forgetting eval() before evaluation is a classic bug.
5. **no_grad() 关梯度追踪**省内存提速; 和 eval() 独立, 评估时都要。
   no_grad() disables tracking to save memory/time; separate from eval(), use both at eval.

### 下一节 / Next
**9.4 Keras 入门**——另一大主流框架。用 Keras 3(可跑在 torch 后端) 看 Sequential/Functional API, 对比它和 PyTorch 的设计哲学。
**9.4 Keras Basics** — the other major framework. Keras 3 (running on the torch backend) with its Sequential/Functional API, contrasting its design philosophy with PyTorch's.
